# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zezo-Elkafoury/Flyrank-internship-assignment-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the performance of a single content page for one reporting day within a specific month. Each row contains historical search and content signals that are available before a refresh decision is made.

This notebook uses May 2026 as the development window because it is a historical mid-panel month. Following the warehouse guidance, the final month is avoided during feature development to reduce the risk of leaking future information into the modeling process.



In [19]:
from google.colab import userdata
userdata.get('HF_TOKEN')
from datasets import load_dataset
df = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [27]:
# To preview data from a streaming dataset, take a few items and convert them to a DataFrame.
import pandas as pd

# Define sample size and shuffle buffer size
sample_size = 15000
shuffle_buffer_size = 20000

# Take a random sample from the streaming dataset
# First, shuffle the dataset, then take the desired number of samples
random_sample_items = list(df.shuffle(seed=42, buffer_size=shuffle_buffer_size).take(sample_size))

# Convert to a pandas DataFrame for preview
sample_df = pd.DataFrame(random_sample_items)
sample_df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-05-01,client_3ffa76342f366962,content_9c6d5fbe262f484a,True,True,False,False,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2026-05-01,client_e547b89c05043229,content_be6cc01d7558cb11,True,True,True,False,2,0,90,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-05-01,client_3ffa76342f366962,content_96a645eca06e62f9,True,True,False,False,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-05-01,client_3ffa76342f366962,content_435036a02dbd46d1,True,True,False,False,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2026-05-01,client_3ffa76342f366962,content_1e895bc3af6b01be,True,True,False,False,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   report_date               15000 non-null  object 
 1   client_hash_id            15000 non-null  object 
 2   content_hash_id           15000 non-null  object 
 3   client_has_gsc            15000 non-null  bool   
 4   client_has_ga4            15000 non-null  bool   
 5   gsc_data_available        15000 non-null  bool   
 6   ga4_data_available        13612 non-null  object 
 7   gsc_impressions           15000 non-null  int64  
 8   gsc_clicks                15000 non-null  int64  
 9   gsc_sum_position          15000 non-null  int64  
 10  gsc_avg_position          5422 non-null   float64
 11  ga4_pageviews             13612 non-null  float64
 12  ga4_sessions              13612 non-null  float64
 13  ga4_users                 13612 non-null  float64
 14  ga4_en

In [29]:
sample_df['report_date'] = pd.to_datetime(sample_df['report_date'])
sample_df['month'] = sample_df['report_date'].dt.strftime('%Y-%m')
sample_df['month'].unique()

array(['2026-05'], dtype=object)

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Filter the development month
may_df = sample_df[sample_df["month"] == "2026-05"]

# Verify the time window
print("Date range:")
print(f"Start: {may_df['report_date'].min()}")
print(f"End:   {may_df['report_date'].max()}")

print("\nNumber of rows:")
print(len(may_df))

# Verify the grain (unit of analysis)
duplicates = may_df.duplicated(
    subset=["client_hash_id", "content_hash_id", "report_date"]
).sum()

print("\nDuplicate (client, content, date) combinations:", duplicates)

if duplicates == 0:
    print("Each row represents one content page on one reporting day.")
else:
    print("Duplicate rows found. Check the dataset grain.")


Date range:
Start: 2026-05-01 00:00:00
End:   2026-05-01 00:00:00

Number of rows:
15000

Duplicate (client, content, date) combinations: 0
Each row represents one content page on one reporting day.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- Features: gsc_impressions, gsc_clicks, gsc_sum_position, ga4_pageviews, gsc_avg_position

- Label: (Proxy Target) Refresh Priority Score. The warehouse does not contain a direct "refresh priority" label. Instead, the goal is to estimate a priority score from historical performance signals so that content pages can be ranked by their expected need for review.

- Context: report_date, client_hash_id, content_hash_id

- Excluded: client_hash_id (as a feature), content_hash_id (as a feature),
Any future observations, Any label-derived columns. They would not be available at the moment the refresh decision is made and would cause data leakage.




In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

#### Verify the grain (One row = one content page on one reporting day)

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check whether each (client, content, date) combination is unique

duplicates = may_df.duplicated(
    subset=["client_hash_id", "content_hash_id", "report_date"]
).sum()

print(f"Duplicate rows: {duplicates}")

if duplicates == 0:
    print("✅ Verified: each row represents one content page on one reporting day.")
else:
    print("⚠️ Duplicate combinations found.")

Duplicate rows: 0
✅ Verified: each row represents one content page on one reporting day.


#### Verify the time window

In [36]:
print(f"Rows in March: {len(may_df)}")
print(f"Start: {may_df['report_date'].min()}")
print(f"End:   {may_df['report_date'].max()}")

Rows in March: 15000
Start: 2026-05-01 00:00:00
End:   2026-05-01 00:00:00


Note: The range is not right here due to the small sample size as loading the full data will be expensive and computationally expensive

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- This dataset captures historical search performance but does not directly measure the business impact of refreshing a page. As a result, the Refresh Priority Score is a decision-support proxy rather than a direct measure of refresh success.

- The analysis uses a historical snapshot and therefore may not capture longer-term seasonal changes or future shifts in search behavior, and the model may need future enhancements

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.